# Feature Engineering & ML Model Comparison
Run all cells from the project root. Chronological train/validation/test split; validation selects the model and the final test stays untouched until selection. Directional accuracy is not conventional regression accuracy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import load_data
from src.features import build_feature_frame
modeling, returns, metadata = load_data(ROOT / 'data')
frames = build_feature_frame(modeling, returns.columns)
print('Assets:', len(frames), 'Dates:', len(modeling))

## Prepare features and chronological partitions
All features are observed at time t, and the target is the return at t+1. Rows are split by date (not randomly).

In [ ]:
data = pd.concat([frame.dropna(subset=['target']).assign(Date=frame.index) for frame in frames.values()], ignore_index=True)
data = data.replace([np.inf, -np.inf], np.nan)
feature_cols = [c for c in data if c not in ('target', 'Date')]
cut1, cut2 = data['Date'].quantile([0.65, 0.82]).tolist()
train = data[data.Date < cut1].copy()
valid = data[(data.Date >= cut1) & (data.Date < cut2)].copy()
test = data[data.Date >= cut2].copy()
print('Train/Validation/Test:', len(train), len(valid), len(test))
print('Cutoffs:', cut1, cut2)
print('Engineered features:', len(feature_cols)-1)

## Compare 5 algorithms with feature selection
Feature selection uses training data only. The model with lowest validation MAE is chosen; directional accuracy is also shown. Sampling is chronological and reproducible for reasonable training time.

In [ ]:
numeric = [c for c in feature_cols if c != 'ticker']
prep = ColumnTransformer([
 ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler()), ('select', SelectKBest(f_regression, k=min(22,len(numeric))))]), numeric),
 ('ticker', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['ticker'])
])
models = {
 'Ridge': Ridge(alpha=100.0),
 'Elastic Net': ElasticNet(alpha=0.0001, l1_ratio=0.3, max_iter=1500),
 'Random Forest': RandomForestRegressor(n_estimators=75, max_depth=9, min_samples_leaf=12, n_jobs=-1, random_state=42),
 'Extra Trees': ExtraTreesRegressor(n_estimators=75, max_depth=10, min_samples_leaf=12, n_jobs=-1, random_state=42),
 'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=15, l2_regularization=10, random_state=42)
}
# Same training subset for every model, retaining dates in chronological order.
train_sample = train.sample(n=min(55000, len(train)), random_state=42).sort_values('Date')
valid_sample = valid.sample(n=min(20000, len(valid)), random_state=42)
results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', prep), ('regressor', estimator)])
    pipe.fit(train_sample[feature_cols], train_sample.target)
    pred = pipe.predict(valid_sample[feature_cols])
    results.append({'Model':name, 'MAE':mean_absolute_error(valid_sample.target,pred), 'RMSE':np.sqrt(mean_squared_error(valid_sample.target,pred)), 'R2':r2_score(valid_sample.target,pred), 'Directional Accuracy %':100*np.mean((pred>0)==(valid_sample.target.to_numpy()>0))})
comparison = pd.DataFrame(results).sort_values('MAE')
display(comparison)
best_name = comparison.iloc[0]['Model']
print('Selected on validation MAE:', best_name)

## Untouched test evaluation and selected feature ranking
Test performance is reported once after model selection. Baseline predicts zero daily return.

In [ ]:
development = pd.concat([train, valid], ignore_index=True)
development_sample = development.sample(n=min(70000,len(development)), random_state=42)
best_pipeline = Pipeline([('preprocess', prep), ('regressor', models[best_name])])
best_pipeline.fit(development_sample[feature_cols], development_sample.target)
yhat = best_pipeline.predict(test[feature_cols])
metrics = {'MAE':float(mean_absolute_error(test.target,yhat)), 'RMSE':float(np.sqrt(mean_squared_error(test.target,yhat))), 'R2':float(r2_score(test.target,yhat)), 'Directional Accuracy %':float(100*np.mean((yhat>0)==(test.target.to_numpy()>0)))}
print('Final test:', best_name, metrics)
print('Zero-return baseline MAE:', mean_absolute_error(test.target,np.zeros(len(test))))
selected = best_pipeline.named_steps['preprocess'].named_transformers_['numeric'].named_steps['select'].get_support()
print('Selected numeric features:', [name for name, keep in zip(numeric, selected) if keep])

## Save the best model for the Streamlit application
The website loads this model automatically; no changes to the four-page navigation are needed.

In [ ]:
out = ROOT / 'models'
out.mkdir(exist_ok=True)
joblib.dump({'pipeline':best_pipeline, 'name':best_name, 'features':feature_cols, 'metrics':metrics}, out/'best_model.joblib')
comparison.to_csv(out/'validation_model_comparison.csv', index=False)
pd.DataFrame([{'Model':best_name, **metrics}]).to_csv(out/'final_test_metrics.csv', index=False)
print('Saved:', out/'best_model.joblib')